# Healthcare FAQ Chatbot: A Simple Local RAG Project

This project explores a practical question: **Can a chatbot answer basic clinic questions using a small set of supplied facts?** I started with six fictional FAQs about appointments, first visits, parking, billing, and medical records. The goal was to build a small workflow that I could inspect and explain, from loading the data to checking the final answer.

The notebook uses **retrieval-augmented generation (RAG)**. Chroma searches the FAQ collection for relevant evidence, and a local language model uses that evidence to write an answer. The model is not being trained on the FAQs. Instead, the relevant text is included in each request.

## What this version includes

| Component | Role in the project |
|---|---|
| `healthcare_faqs.json` | Six fictional clinic FAQs, each with an ID, topic, question, answer, and source label |
| Chroma's default embedding function | Represents FAQ text and questions numerically for semantic search |
| ChromaDB | Stores the evidence and retrieves the closest matches |
| Ollama | Runs the language model locally |
| `llama3.2:3b` | Generates answers in the final workflow |
| Manual review | Checks whether retrieval and generation worked separately |

This is a portfolio learning demo using administrative information. The source labels refer to fictional guides, not published hospital policies. There are no patient records in this dataset, and the chatbot is not intended for diagnosis or treatment advice.

## Before running the notebook

Keep `healthcare_faqs.json` beside this notebook and open the project folder in VS Code. Select the project's Python environment as the notebook kernel. Install the Python packages in that environment using the terminal:

```bash
python -m pip install chromadb ollama ipykernel
```

Install and open Ollama separately. The notebook retains one small-model connection test and one diagnostic example, so a complete run needs both model downloads:

```bash
ollama pull llama3.2:1b
ollama pull llama3.2:3b
```

Run cells from top to bottom. The first embedding request may download model files, and the first generation request may take longer while Ollama loads a model. The local `chroma_db` folder is generated by the notebook and can be rebuilt from the JSON file.

**About the saved outputs:** the outputs below come from the uploaded notebook's earlier runs. They have been preserved, including the unsuccessful booking example. This documentation edit has not rerun local Ollama inference. Future answers may vary with software, model versions, and hardware.

## 1. Load and inspect the FAQ data

The first step is to make sure Python can read the source file. `json.load()` turns the JSON content into a list of dictionaries, so each FAQ's fields can be accessed by name.

Printing all six entries gives me a chance to check the actual facts before using them as evidence. If a source answer is missing or incorrect, retrieval and generation cannot reliably fix it. `Path("healthcare_faqs.json")` looks in the notebook's current working folder; a file-not-found error usually means the project folder or file location needs checking.

The saved output shows six FAQs loaded successfully. These are intentionally short, easy-to-check examples rather than a comprehensive clinic knowledge base.

In [20]:
import json
from pathlib import Path

# Find the FAQ file in the current working folder.
faq_path = Path("healthcare_faqs.json")

# Load our FAQ evidence.
with open(faq_path, "r", encoding="utf-8") as file:
    faqs = json.load(file)

print(f"Loaded {len(faqs)} FAQs.\n")

for faq in faqs:
    print(f"ID: {faq['id']}")
    print(f"Question: {faq['question']}")
    print(f"Answer: {faq['answer']}")
    print(f"Source: {faq['source']}")
    print()

Loaded 6 FAQs.

ID: faq_001
Question: How do I schedule an appointment?
Answer: Appointments can be scheduled through the patient portal or by calling the scheduling office Monday through Friday, from 8 AM to 5 PM.
Source: Fictional clinic appointment guide

ID: faq_002
Question: How do I cancel an appointment?
Answer: Cancel through the patient portal or call the scheduling office. Please provide at least 24 hours of notice.
Source: Fictional clinic appointment guide

ID: faq_003
Question: What should I bring to my first appointment?
Answer: Bring a photo ID, your insurance card, and a current list of medications. Arrive 15 minutes before your scheduled appointment.
Source: Fictional clinic visitor guide

ID: faq_004
Question: Where can patients park?
Answer: Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.
Source: Fictional clinic visitor guide

ID: faq_005
Question: Who can help me understand my bill?
Answe

## 2. Prepare the documents and metadata

Chroma needs matching lists of IDs, document text, and metadata. The same list position must refer to the same FAQ in all three lists.

I combine each FAQ's question and answer into its searchable text. The question adds useful wording about the topic, while the answer contains the facts needed for generation. The topic and source label stay attached as metadata so the retrieved result can be identified later.

**One FAQ is one chunk in this version.** Each entry is short and covers one main subject, so splitting it further would add complexity without a clear benefit. For example, the first-visit FAQ keeps its arrival time and items to bring together.

In [21]:
# Unique labels for each FAQ.
ids = [faq["id"] for faq in faqs]

# Text the chatbot will search for evidence.
documents = [
    f"Question: {faq['question']}\nAnswer: {faq['answer']}"
    for faq in faqs
]

# Details we can use to identify and display each source.
metadatas = [
    {
        "topic": faq["topic"],
        "source": faq["source"]
    }
    for faq in faqs
]

print(f"Prepared {len(documents)} documents.")
print("\nFirst document:")
print(documents[0])
print("\nIts metadata:")
print(metadatas[0])

Prepared 6 documents.

First document:
Question: How do I schedule an appointment?
Answer: Appointments can be scheduled through the patient portal or by calling the scheduling office Monday through Friday, from 8 AM to 5 PM.

Its metadata:
{'topic': 'appointments', 'source': 'Fictional clinic appointment guide'}


## 3. Create the persistent Chroma collection

This cell opens a local database in `chroma_db` and creates or reopens the `healthcare_faqs` collection. Persistence means the stored evidence can be reused after the notebook closes, including by the separate Streamlit app.

Because I supply documents without embedding vectors, Chroma uses its default embedding function to create them. The embedding model supports retrieval; it is separate from the Llama model used later to write answers.

`upsert()` adds a new ID or updates an existing one. This makes rerunning the indexing cell safe from duplicate IDs. It does **not** delete records removed from the JSON file; a changed dataset would need a deliberate cleanup or rebuild. For this unchanged six-FAQ collection, the saved count is six.

In [22]:
import chromadb

# Save the database in our project folder.
client = chromadb.PersistentClient(path="chroma_db")

# Create a collection, or open it if it already exists.
collection = client.get_or_create_collection(
    name="healthcare_faqs"
)

# Store the FAQ text, IDs, and metadata.
# Chroma automatically creates embeddings for the documents.
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f"Collection ready! Stored {collection.count()} FAQs.")

Collection ready! Stored 6 FAQs.


## 4. Test retrieval with different wording

The question “Can I leave my car there for free?” does not copy the FAQ question “Where can patients park?” This makes it a useful small test of searching by meaning rather than exact wording.

Here I request **top-k = 3**, which returns the three closest FAQs. The saved output ranks `faq_004`, the parking FAQ, first. That is the evidence needed to answer the question.

The next two results cover cancellation and billing. They are not useful answers to the parking question, even though Chroma returned them. This illustrates an important limitation: **a closest match is not automatically relevant or sufficient evidence.** At this stage, no language model has generated an answer.

In [23]:
question = "Can I leave my car there for free?"

# Retrieve the three most similar FAQs.
results = collection.query(
    query_texts=[question],
    n_results=3
)

print(f"Question: {question}\n")

for rank, (faq_id, document, metadata) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0]
    ),
    start=1
):
    print(f"Result {rank}: {faq_id}")
    print(document)
    print(f"Source: {metadata['source']}")
    print()

Question: Can I leave my car there for free?

Result 1: faq_004
Question: Where can patients park?
Answer: Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.
Source: Fictional clinic visitor guide

Result 2: faq_002
Question: How do I cancel an appointment?
Answer: Cancel through the patient portal or call the scheduling office. Please provide at least 24 hours of notice.
Source: Fictional clinic appointment guide

Result 3: faq_005
Question: Who can help me understand my bill?
Answer: Contact the billing office through the patient portal. Billing staff are available Monday through Friday, from 9 AM to 4 PM.
Source: Fictional clinic billing guide



## 5. Assemble the retrieved context

`context` combines the retrieved passages into one text block. Each passage includes its FAQ ID and source label so the evidence can be traced back to the original entry.

This demonstrates the handoff between retrieval and generation: the retriever supplies text that can be included in a model prompt. Adding labels makes citation possible, but does not guarantee the model will cite or use them correctly.

The final function later in the notebook uses only the top result to keep the first version simple. This earlier three-passage context remains useful for understanding how evidence is assembled.

In [24]:
# Label each retrieved passage so the model can cite its FAQ ID.
context = "\n\n".join(
    f"[{faq_id}]\n{document}\nSource: {metadata['source']}"
    for faq_id, document, metadata in zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0]
    )
)

print(context)

[faq_004]
Question: Where can patients park?
Answer: Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.
Source: Fictional clinic visitor guide

[faq_002]
Question: How do I cancel an appointment?
Answer: Cancel through the patient portal or call the scheduling office. Please provide at least 24 hours of notice.
Source: Fictional clinic appointment guide

[faq_005]
Question: Who can help me understand my bill?
Answer: Contact the billing office through the patient portal. Billing staff are available Monday through Friday, from 9 AM to 4 PM.
Source: Fictional clinic billing guide


## 6. Install the Ollama Python client

Ollama has two parts in this setup: the application that runs the model on my computer, and the Python client that lets the notebook communicate with it. This cell installs the Python client into the active notebook environment.

Installing this package does not install the Ollama application or download a language model. Those steps are handled separately using the setup instructions above. Routine package-installation notices have been removed from the saved output to keep this notebook readable.

In [25]:
%pip install -q ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 7. Check the local model connection

Before combining retrieval with generation, I send a basic greeting request to `llama3.2:1b`. The saved response, “Hello,” confirms that Python could communicate with Ollama and receive generated text.

This is only a connection test. A model answering a greeting does not establish that it can follow grounding instructions or answer FAQ questions accurately. The final chatbot uses `llama3.2:3b` after problems appeared during the smaller-model experiments.

In [26]:
import ollama

response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ]
)

print(response["message"]["content"])

Hello.


## 8. Inspect the exact evidence during troubleshooting

During development, the smaller model sometimes refused to answer the parking question even when the correct FAQ was available. This diagnostic step checks the text itself instead of assuming that retrieval was the cause.

The definitions below select the parking FAQ manually for this inspection. `repr()` displays line breaks as `\n`, making the string structure visible. The saved output includes the complete parking answer: free patient parking in Lot A beside the main entrance.

This manual selection is a debugging tool, not the final retrieval method. It helps distinguish missing evidence from a model that fails to use available evidence.

In [27]:
parking_faq = next(faq for faq in faqs if faq['id'] == 'faq_004')
parking_context = (
    f"[{parking_faq['id']}]\n"
    f"Question: {parking_faq['question']}\n"
    f"Answer: {parking_faq['answer']}"
)

print("EXACT PARKING EVIDENCE:")
print(repr(parking_context))

print("\nPARKING FAQ FROM OUR DATA:")
print(parking_faq)

EXACT PARKING EVIDENCE:
'[faq_004]\nQuestion: Where can patients park?\nAnswer: Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.'

PARKING FAQ FROM OUR DATA:
{'id': 'faq_004', 'topic': 'parking', 'question': 'Where can patients park?', 'answer': 'Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.', 'source': 'Fictional clinic visitor guide'}


## 9. Try a direct reading-comprehension check

Here the model receives only the parking answer text and a direct question about the parking location. It correctly identifies Lot A beside the main entrance in the saved output.

This shows that the smaller model could use the passage under this particular prompt. It does not prove that the earlier refusals had one specific cause: the wording, evidence format, and instructions differed across the development experiments. I treat this as a diagnostic observation rather than a controlled comparison.

In [28]:
response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": (
                f"Read this passage:\n{parking_faq['answer']}\n\n"
                "According to the passage, where is free patient parking?"
            )
        }
    ],
    options={"temperature": 0}
)

print(response["message"]["content"])

According to the passage, free patient parking is available in Lot A beside the main entrance.


## 10. Generate an answer after automatic retrieval

This cell restores automatic retrieval and uses `llama3.2:3b` for generation. Chroma selects the top FAQ, then the prompt includes that passage and the original parking question.

The saved run retrieved `faq_004` and answered, “Yes, free patient parking is available.” That claim is supported by the source. It does not repeat the location, but it answers the question about cost.

Using **top-k = 1** keeps the input short and avoids unrelated passages in this first version. The tradeoff is that questions needing information from more than one FAQ may not receive enough context. One successful response also does not establish that the larger model is reliable across all questions.

In [29]:
question = "Can I leave my car there for free?"

# Retrieve the closest FAQ automatically.
results = collection.query(
    query_texts=[question],
    n_results=1
)

retrieved_text = results["documents"][0][0]
retrieved_id = results["ids"][0][0]

# Keep the same prompt; change only the model.
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": (
                f"Read this passage:\n{retrieved_text}\n\n"
                f"According to the passage, {question}\n"
                "Answer briefly using only the passage."
            )
        }
    ],
    options={"temperature": 0}
)

print("Retrieved FAQ:", retrieved_id)
print("\nAnswer:")
print(response["message"]["content"])

Retrieved FAQ: faq_004

Answer:
Yes, free patient parking is available.


## 11. Put the workflow into a reusable function

`ask_chatbot()` combines the steps into one call: retrieve the closest FAQ, send its text and the question to the local model, and return the answer with its evidence.

The prompt asks the model to stay within the supplied passage and acknowledge missing information. These instructions guide behavior; they are not a guarantee. Setting temperature to zero reduces sampling randomness, but does not make answers automatically correct or guarantee identical results everywhere.

The returned dictionary contains the answer, retrieved FAQ ID, source label, and full evidence. Keeping these together makes review easier. A displayed retrieved source is not the same as a verified citation: I still need to check that it actually supports the generated claim.

Each call is independent. This function does not remember earlier questions or resolve follow-ups such as “What about tomorrow?”

In [30]:
def ask_chatbot(question):
    # Find the closest FAQ.
    results = collection.query(
        query_texts=[question],
        n_results=1
    )

    passage = results["documents"][0][0]
    faq_id = results["ids"][0][0]
    source = results["metadatas"][0][0]["source"]

    # Generate an answer from the retrieved evidence.
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": (
                    f"Read this passage:\n{passage}\n\n"
                    f"According to the passage, {question}\n"
                    "Answer briefly using only the passage.\n"
                    "If the passage does not contain the answer, "
                    "say 'The FAQs do not provide that information.'"
                )
            }
        ],
        options={"temperature": 0}
    )

    return {
        "answer": response["message"]["content"],
        "retrieved_id": faq_id,
        "source": source,
        "evidence": passage
    }

## 12. Check a first-visit question

This test asks about arrival time using different wording from the source question. Chroma retrieves `faq_003`, which includes both what to bring and when to arrive.

The saved answer says to arrive 15 minutes before the scheduled appointment. That matches the evidence. Displaying the answer and passage together lets me check retrieval and generation separately instead of judging the response only by how natural it sounds.

In [31]:
result = ask_chatbot("How early should I arrive for my first visit?")

print("Answer:", result["answer"])
print("\nRetrieved source:", result["retrieved_id"], "-", result["source"])
print("\nEvidence:", result["evidence"])

Answer: Arrive 15 minutes before your scheduled appointment.

Retrieved source: faq_003 - Fictional clinic visitor guide

Evidence: Question: What should I bring to my first appointment?
Answer: Bring a photo ID, your insurance card, and a current list of medications. Arrive 15 minutes before your scheduled appointment.


## 13. Check a question with no supported answer

None of the six FAQs contains a Wi-Fi password. This test checks whether the model recognizes that the evidence is insufficient.

Chroma still returns `faq_004` because the query requests one nearest match. The saved answer says the passage does not contain the Wi-Fi password, which is appropriate. The wording differs from the suggested fallback sentence, but the meaning is correct.

This is one successful abstention example. The current workflow does not use a calibrated relevance threshold or an independent answer-verification step, so it still relies on the model to recognize when a passage cannot answer a question.

In [32]:
result = ask_chatbot("What is the Wi-Fi password at the clinic?")

print("Answer:", result["answer"])
print("\nRetrieved FAQ:", result["retrieved_id"])
print("\nEvidence:", result["evidence"])

Answer: The passage does not contain the Wi-Fi password.

Retrieved FAQ: faq_004

Evidence: Question: Where can patients park?
Answer: Free patient parking is available in Lot A beside the main entrance. Accessible parking spaces are located near the entrance.


## 14. Review several FAQ topics

The final loop tests booking, cancellation, billing hours, and records turnaround. These are manually written development checks rather than a large, held-out evaluation set.

The saved outputs show a useful contrast:

| Question topic | Retrieved FAQ | Review of the saved answer |
|---|---|---|
| Booking | `faq_001` | Incorrect refusal; the passage explains portal and phone scheduling |
| Cancellation | `faq_002` | Correct: at least 24 hours of notice |
| Billing | `faq_005` | Correct: Monday–Friday, 9 AM–4 PM |
| Medical records | `faq_006` | Correct: normally within 7 business days |

All four expected FAQs were retrieved first in this small batch, but only three generated answers were correct. The booking result is an **unnecessary refusal**, not a missing-data problem: the model had the answer and failed to use it. I kept this result because it shows why retrieval quality and answer quality need separate checks.

In [33]:
test_questions = [
    "How can I book an appointment?",
    "How much notice should I give when canceling?",
    "When is the billing office available?",
    "How long does a medical records request take?"
]

for question in test_questions:
    result = ask_chatbot(question)

    print("Question:", question)
    print("Answer:", result["answer"])
    print("Retrieved FAQ:", result["retrieved_id"])
    print("-" * 50)

Question: How can I book an appointment?
Answer: The FAQs do not provide that information.
Retrieved FAQ: faq_001
--------------------------------------------------
Question: How much notice should I give when canceling?
Answer: At least 24 hours of notice.
Retrieved FAQ: faq_002
--------------------------------------------------
Question: When is the billing office available?
Answer: Monday through Friday, from 9 AM to 4 PM.
Retrieved FAQ: faq_005
--------------------------------------------------
Question: How long does a medical records request take?
Answer: Requests are normally processed within 7 business days.
Retrieved FAQ: faq_006
--------------------------------------------------


## What I learned and what this version leaves open

The main result is a working local RAG workflow that can retrieve fictional clinic facts and generate short answers. The parking and first-visit examples show the pieces working together. The Wi-Fi example shows a case where the model correctly acknowledges missing information.

The most useful lesson was that **finding the right evidence and using it correctly are separate problems**. The booking test retrieved the correct FAQ but still produced an incorrect refusal. Earlier experiments also showed that prompt wording and model choice can affect behavior. I would not describe this version as hallucination-free or fully reliable.

The project is intentionally small. Six FAQs make the workflow easy to inspect, but they do not represent the variety of questions a real clinic would receive. Top-1 retrieval can miss information needed from another FAQ, source labels are fictional, and the current tests are too limited to estimate real-world accuracy. Questions about overnight parking, specific insurance coverage, or other details absent from the sources should not be answered by assumption.

### How the notebook connects to the webpage

The separate `app.py` Streamlit interface opens the same `healthcare_faqs` collection in `chroma_db`, calls the local model, and displays the retrieved evidence in an expandable panel. The notebook documents the data preparation and experiments; the app provides a simple way to try the workflow. The webpage code is not embedded in this notebook.

### Preparing this version for GitHub

Include this notebook, `healthcare_faqs.json`, `app.py`, a requirements file, and a README with setup instructions. Exclude the virtual environment and generated database from version control. Someone cloning the repository should install the dependencies, download the Ollama models, run this notebook to build the database, and then start the interface:

```bash
python -m streamlit run app.py
```

Before publishing, restart the kernel and run the notebook from top to bottom on the machine running Ollama. Check the current answers against the JSON evidence, record the package and model versions used, and keep the known booking limitation visible. The saved outputs here document the submitted development runs; they should not be presented as a newly executed verification.

For a later version, I would use a larger fixed question set, include paraphrases and unsupported questions, and record retrieval success, factual support, and unnecessary refusals separately. Those additions would help measure improvements without making this first demo harder to understand.